# RideSmart - modelo alinhado ao enunciado do professor

Este notebook implementa a modelagem pedida no projeto final da disciplina.

O problema considera:

- **A**: ponto inicial do passageiro;
- **B**: destino final;
- **X**: distância máxima que o passageiro aceita caminhar;
- **P**: ponto de embarque escolhido dentro do raio de caminhada.

A viagem é modelada em dois trechos:

```text
A -> P: caminhada no grafo de pedestres
P -> B: carro no grafo de direção
```

Não existe posição inicial do veículo antes de **P** neste notebook. O carro aparece apenas a partir do ponto de embarque **P**, exatamente para manter a modelagem fiel ao enunciado.

O objetivo é comparar algoritmos de caminho mínimo e pesos diferentes para descobrir se permitir caminhada até **P** melhora a rota em relação ao caso sem caminhada, isto é, **P = A**.

Os algoritmos comparados são:

- Dijkstra simples;
- Dijkstra com heap;
- A*;
- Dijkstra bidirecional.

Os critérios avaliados são:

- menor distância;
- menor tempo sem trânsito;
- menor tempo com trânsito sintético;
- caso sem caminhada;
- ganho ao caminhar.

## 1. Instalação das bibliotecas

Esta célula prepara o ambiente.

Antes de instalar qualquer pacote, o código verifica se a biblioteca já existe. Isso evita reinstalações desnecessárias em ambientes locais ou no Colab.

Bibliotecas usadas:

- `osmnx`: baixa e manipula dados reais do OpenStreetMap;
- `networkx`: estrutura de grafos e apoio aos algoritmos;
- `folium`: mapas interativos;
- `pandas`: tabelas de candidatos, resultados e ganhos.

In [1]:
import importlib.util
import subprocess
import sys

pacotes = {
    "osmnx": "osmnx",
    "networkx": "networkx",
    "folium": "folium",
    "pandas": "pandas",
}

faltando = [pip_name for import_name, pip_name in pacotes.items() if importlib.util.find_spec(import_name) is None]

if faltando:
    print("Instalando pacotes:", faltando)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltando])
else:
    print("Todas as bibliotecas necessárias já estão instaladas.")

Todas as bibliotecas necessárias já estão instaladas.


## 2. Importações e pasta de saída

Esta célula carrega as bibliotecas e cria a pasta de saída.

Todos os mapas e tabelas gerados por este notebook serão salvos em `saidas_modelo_professor`.

Também ativamos o cache do OSMnx. Assim, quando a mesma cidade for baixada novamente, a execução tende a ficar mais rápida.

In [2]:
from pathlib import Path
import heapq
import math
import random
import time

import folium
from folium import plugins
import networkx as nx
import osmnx as ox
import pandas as pd
from IPython.display import display

ox.settings.use_cache = True
ox.settings.log_console = False

PASTA_SAIDA = Path("saidas_modelo_professor")
PASTA_SAIDA.mkdir(exist_ok=True)

print("Bibliotecas carregadas.")

Bibliotecas carregadas.


## 3. Dados de entrada

Aqui são definidos os dados do cenário.

O usuário informa:

- `CIDADE`: área que será baixada do OpenStreetMap;
- `PONTO_A`: local inicial do passageiro;
- `PONTO_B`: destino final.

Os pontos são passados por nome/endereço textual. O OSMnx fará a geocodificação para encontrar latitude e longitude.

Para reduzir ambiguidade, escreva cidade, estado e país nos endereços.

In [3]:
CIDADE = "Natal, RN, Brasil"

PONTO_A = "Midway Mall, Natal, RN, Brasil"
PONTO_B = "Agaé, Natal, RN, Brasil"

print("Cidade:", CIDADE)
print("Ponto A:", PONTO_A)
print("Ponto B:", PONTO_B)

Cidade: Natal, RN, Brasil
Ponto A: Midway Mall, Natal, RN, Brasil
Ponto B: Agaé, Natal, RN, Brasil


## 4. Parâmetros do modelo

Esta célula define os parâmetros que controlam a análise.

- `RAIO_CAMINHADA_M`: limite máximo de caminhada do passageiro;
- `VELOCIDADE_CAMINHADA_KMH`: velocidade média usada para converter distância a pé em tempo;
- `DISTANCIA_MAX_EMBARQUE_CARRO_M`: distância máxima aceitável entre o ponto caminhável **P** e um nó da malha de carros;
- `SEMENTE_ALEATORIA`: fixa o sorteio do trânsito sintético, tornando o experimento reprodutível.

Um candidato **P** só será usado pelos algoritmos se:

```text
D_walk(A, P) <= X
```

e se houver um ponto de carro suficientemente próximo para representar o local de embarque.

In [4]:
RAIO_CAMINHADA_M = 500
VELOCIDADE_CAMINHADA_KMH = 5.0
DISTANCIA_MAX_EMBARQUE_CARRO_M = 25
SEMENTE_ALEATORIA = 42

print(f"Raio máximo de caminhada X: {RAIO_CAMINHADA_M} m")
print(f"Velocidade média de caminhada: {VELOCIDADE_CAMINHADA_KMH} km/h")
print(f"Distância máxima entre P a pé e embarque de carro: {DISTANCIA_MAX_EMBARQUE_CARRO_M} m")

Raio máximo de caminhada X: 500 m
Velocidade média de caminhada: 5.0 km/h
Distância máxima entre P a pé e embarque de carro: 25 m


## 5. Carregar ou baixar a cidade em dois grafos

O notebook usa dois grafos porque pedestres e carros não possuem exatamente a mesma malha.

- `G_caminhada`: `network_type="walk"`, usado no trecho **A -> P**;
- `G_direcao`: `network_type="drive"`, usado no trecho **P -> B**.

Essa separação evita uma simplificação ruim: permitir que o carro passe por caminhos de pedestre ou obrigar o pedestre a seguir apenas ruas de carro.

Para acelerar testes, esta célula usa cache local em disco:

```text
cache_grafos/
```

Se os grafos da cidade já existirem nessa pasta, o notebook carrega os arquivos `.graphml`. Se ainda não existirem, ele baixa do OpenStreetMap, processa os pesos e salva os grafos para reutilização.

Depois do download, o grafo de direção recebe pesos de velocidade e tempo:

```text
travel_time = length / velocidade_estimada
```

O peso `length` fica em metros e `travel_time` fica em segundos.

In [5]:
def nome_seguro_cache(texto):
    permitido = []
    for caractere in texto.lower():
        if caractere.isalnum():
            permitido.append(caractere)
        else:
            permitido.append("_")
    nome = "".join(permitido)
    while "__" in nome:
        nome = nome.replace("__", "_")
    return nome.strip("_")


PASTA_CACHE_GRAFOS = Path("cache_grafos")
PASTA_CACHE_GRAFOS.mkdir(exist_ok=True)

prefixo_cache = nome_seguro_cache(CIDADE)
arquivo_cache_walk = PASTA_CACHE_GRAFOS / f"{prefixo_cache}_walk.graphml"
arquivo_cache_drive = PASTA_CACHE_GRAFOS / f"{prefixo_cache}_drive.graphml"

if arquivo_cache_walk.exists() and arquivo_cache_drive.exists():
    print("Carregando grafos do cache local...")
    print(f"- {arquivo_cache_walk}")
    print(f"- {arquivo_cache_drive}")
    G_caminhada = ox.load_graphml(arquivo_cache_walk)
    G_direcao = ox.load_graphml(arquivo_cache_drive)
else:
    print("Cache não encontrado. Baixando grafo de caminhada...")
    G_caminhada = ox.graph_from_place(CIDADE, network_type="walk", simplify=True)

    print("Baixando grafo de direção...")
    G_direcao = ox.graph_from_place(CIDADE, network_type="drive", simplify=True)

    if hasattr(ox, "add_edge_lengths"):
        G_caminhada = ox.add_edge_lengths(G_caminhada)
        G_direcao = ox.add_edge_lengths(G_direcao)
    else:
        G_caminhada = ox.distance.add_edge_lengths(G_caminhada)
        G_direcao = ox.distance.add_edge_lengths(G_direcao)

    G_direcao = ox.add_edge_speeds(G_direcao)
    G_direcao = ox.add_edge_travel_times(G_direcao)

    print("Salvando grafos no cache local...")
    ox.save_graphml(G_caminhada, arquivo_cache_walk)
    ox.save_graphml(G_direcao, arquivo_cache_drive)

print("Grafo de caminhada:")
print(f"- nós: {len(G_caminhada.nodes):,}")
print(f"- arestas: {len(G_caminhada.edges):,}")

print("\nGrafo de direção:")
print(f"- nós: {len(G_direcao.nodes):,}")
print(f"- arestas: {len(G_direcao.edges):,}")

Carregando grafos do cache local...
- cache_grafos\natal_rn_brasil_walk.graphml
- cache_grafos\natal_rn_brasil_drive.graphml


Grafo de caminhada:
- nós: 26,728
- arestas: 78,120

Grafo de direção:
- nós: 18,578
- arestas: 48,183


## 6. Geocodificar A e B

Esta célula transforma os nomes dos locais em coordenadas geográficas.

Depois, o código encontra os nós mais próximos nos dois grafos:

- no grafo de caminhada, para calcular o trecho **A -> P**;
- no grafo de direção, para calcular o caso sem caminhada e a chegada em **B**.

O ponto **B** usado pelo carro é o nó mais próximo na malha `drive`. O caso sem caminhada usa o nó de carro mais próximo de **A** como ponto de embarque.

In [6]:
lat_A, lon_A = ox.geocode(PONTO_A)
lat_B, lon_B = ox.geocode(PONTO_B)

ROTULO_A = f"A: {PONTO_A}"
ROTULO_B = f"B: {PONTO_B}"

A_caminhada = ox.distance.nearest_nodes(G_caminhada, X=lon_A, Y=lat_A)
B_caminhada = ox.distance.nearest_nodes(G_caminhada, X=lon_B, Y=lat_B)

A_direcao = ox.distance.nearest_nodes(G_direcao, X=lon_A, Y=lat_A)
B_direcao = ox.distance.nearest_nodes(G_direcao, X=lon_B, Y=lat_B)

print("Coordenadas encontradas:")
print(f"A: latitude={lat_A:.6f}, longitude={lon_A:.6f}")
print(f"B: latitude={lat_B:.6f}, longitude={lon_B:.6f}")

print("\nNós mais próximos:")
print(f"A no grafo de caminhada: {A_caminhada}")
print(f"B no grafo de caminhada: {B_caminhada}")
print(f"A no grafo de direção: {A_direcao}")
print(f"B no grafo de direção: {B_direcao}")

Coordenadas encontradas:
A: latitude=-5.812120, longitude=-35.205724
B: latitude=-5.840552, longitude=-35.211156

Nós mais próximos:
A no grafo de caminhada: 567705822
B no grafo de caminhada: 503732242
A no grafo de direção: 501969167
B no grafo de direção: 503732242


## 7. Definir candidatos P

O ponto **P** não é escolhido manualmente.

Primeiro, o notebook busca todos os nós que o passageiro consegue alcançar a pé a partir de **A** dentro do raio **X**:

```text
D_walk(A, P_i) <= X
```

Depois, cada candidato caminhável é associado ao nó mais próximo da malha de carros. Esse nó representa o ponto exato onde o carro poderia iniciar o trecho **P -> B**.

Um candidato só é considerado válido para embarque se:

```text
distância entre P_caminhada e P_carro <= DISTANCIA_MAX_EMBARQUE_CARRO_M
```

O mapa mostra todos os candidatos encontrados, mas os algoritmos só escolhem entre os candidatos válidos.

In [7]:
velocidade_caminhada_mps = VELOCIDADE_CAMINHADA_KMH / 3.6

distancias_caminhada_A = nx.single_source_dijkstra_path_length(
    G_caminhada,
    A_caminhada,
    cutoff=RAIO_CAMINHADA_M,
    weight="length",
)

linhas_candidatos = []

for no_caminhada, distancia_a_pe_m in sorted(distancias_caminhada_A.items(), key=lambda item: item[1]):
    lat_walk = G_caminhada.nodes[no_caminhada]["y"]
    lon_walk = G_caminhada.nodes[no_caminhada]["x"]

    no_direcao = ox.distance.nearest_nodes(G_direcao, X=lon_walk, Y=lat_walk)
    lat_drive = G_direcao.nodes[no_direcao]["y"]
    lon_drive = G_direcao.nodes[no_direcao]["x"]

    distancia_ate_embarque_m = ox.distance.great_circle(lat_walk, lon_walk, lat_drive, lon_drive)
    distancia_total_a_pe_m = distancia_a_pe_m + distancia_ate_embarque_m
    valido_para_embarque = (
        distancia_ate_embarque_m <= DISTANCIA_MAX_EMBARQUE_CARRO_M
        and distancia_total_a_pe_m <= RAIO_CAMINHADA_M
    )

    linhas_candidatos.append({
        "no_caminhada": no_caminhada,
        "no_direcao": no_direcao,
        "distancia_no_grafo_caminhada_m": float(distancia_a_pe_m),
        "distancia_ate_embarque_carro_m": float(distancia_ate_embarque_m),
        "distancia_total_a_pe_m": float(distancia_total_a_pe_m),
        "tempo_caminhada_s": float(distancia_total_a_pe_m / velocidade_caminhada_mps),
        "valido_para_embarque": bool(valido_para_embarque),
        "latitude_pedestre": lat_walk,
        "longitude_pedestre": lon_walk,
        "latitude_embarque": lat_drive,
        "longitude_embarque": lon_drive,
    })

linhas_candidatos = sorted(linhas_candidatos, key=lambda item: item["distancia_total_a_pe_m"])

for i, candidato in enumerate(linhas_candidatos, start=1):
    candidato["id"] = f"P{i}"
    candidato["distancia_no_grafo_caminhada_m"] = round(candidato["distancia_no_grafo_caminhada_m"], 2)
    candidato["distancia_ate_embarque_carro_m"] = round(candidato["distancia_ate_embarque_carro_m"], 2)
    candidato["distancia_total_a_pe_m"] = round(candidato["distancia_total_a_pe_m"], 2)
    candidato["tempo_caminhada_s"] = round(candidato["tempo_caminhada_s"], 2)
    candidato["tempo_caminhada_min"] = round(candidato["tempo_caminhada_s"] / 60, 2)

if not linhas_candidatos:
    raise ValueError("Nenhum candidato P foi encontrado dentro do raio de caminhada.")

df_candidatos_P = pd.DataFrame(linhas_candidatos)
df_candidatos_embarque_P = df_candidatos_P[df_candidatos_P["valido_para_embarque"]].copy()

if df_candidatos_embarque_P.empty:
    raise ValueError(
        "Nenhum candidato P ficou suficientemente próximo da malha de carro. "
        "Aumente DISTANCIA_MAX_EMBARQUE_CARRO_M ou ajuste o ponto A."
    )

print(f"Candidatos alcançáveis a pé dentro de X: {len(df_candidatos_P)}")
print(f"Candidatos válidos para embarque: {len(df_candidatos_embarque_P)}")

df_candidatos_P

Candidatos alcançáveis a pé dentro de X: 139
Candidatos válidos para embarque: 42


,no_caminhada,no_direcao,distancia_no_grafo_caminhada_m,distancia_ate_embarque_carro_m,distancia_total_a_pe_m,tempo_caminhada_s,valido_para_embarque,latitude_pedestre,longitude_pedestre,latitude_embarque,longitude_embarque,id,tempo_caminhada_min
0,567705822,501969167,0.00,72.60,72.60,52.27,False,-5.812194,-35.205540,-5.812700,-35.205126,P1,0.87
1,567705818,501969167,18.26,82.99,101.25,72.90,False,-5.812041,-35.205479,-5.812700,-35.205126,P2,1.22
2,567705807,505552959,61.40,55.79,117.19,84.38,False,-5.812150,-35.206094,-5.812324,-35.206566,P3,1.41
3,2718089713,501969167,61.57,59.14,120.71,86.91,False,-5.812169,-35.205109,-5.812700,-35.205126,P4,1.45
4,567705820,6991058207,40.36,82.36,122.72,88.36,False,-5.811855,-35.205410,-5.811563,-35.204725,P5,1.47
...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,1075237458,505552596,481.64,47.72,529.36,381.14,False,-5.810918,-35.204925,-5.810968,-35.204496,P135,6.35
135,7596242411,526591657,484.10,46.53,530.63,382.06,False,-5.809809,-35.206586,-5.809497,-35.206305,P136,6.37
136,7596102355,529774041,468.14,68.71,536.85,386.53,False,-5.811613,-35.209368,-5.812221,-35.209480,P137,6.44
137,8925498237,529774041,458.74,84.80,543.55,391.35,False,-5.811564,-35.209091,-5.812221,-35.209480,P138,6.52


## 8. Mapa inicial

Este mapa serve para validar visualmente a base espacial antes dos algoritmos.

Ele mostra:

- **A**, ponto inicial do passageiro;
- **B**, destino;
- raio máximo de caminhada **X**;
- candidatos **P**.

Os candidatos em cinza são pontos caminháveis dentro do raio **X**.

Os candidatos em vermelho são pontos caminháveis que também possuem um ponto de embarque próximo na malha de carros. O ícone de carro não aparece neste mapa inicial para evitar poluição visual; ele aparece apenas no mapa final, marcando o ponto real de embarque da rota selecionada.

In [8]:
centro_lat = (lat_A + lat_B) / 2
centro_lon = (lon_A + lon_B) / 2

mapa_inicial = folium.Map(location=[centro_lat, centro_lon], zoom_start=13, tiles="CartoDB positron")

folium.Marker([lat_A, lon_A], popup=ROTULO_A, tooltip="A - passageiro", icon=folium.Icon(color="green", icon="user")).add_to(mapa_inicial)
folium.Marker([lat_B, lon_B], popup=ROTULO_B, tooltip="B - destino", icon=folium.Icon(color="red", icon="flag")).add_to(mapa_inicial)
folium.Circle([lat_A, lon_A], radius=RAIO_CAMINHADA_M, color="#16a34a", fill=False, weight=3, tooltip=f"Raio X: {RAIO_CAMINHADA_M} m").add_to(mapa_inicial)

candidatos_layer = folium.FeatureGroup(name="Candidatos P", show=True)

for _, row in df_candidatos_P.iterrows():
    cor = "#dc2626" if row["valido_para_embarque"] else "#9ca3af"
    raio = 4 if row["valido_para_embarque"] else 2
    status = "válido para embarque" if row["valido_para_embarque"] else "apenas caminhável"
    folium.CircleMarker(
        [row["latitude_pedestre"], row["longitude_pedestre"]],
        radius=raio,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.75,
        tooltip=f"{row['id']} - {status}",
    ).add_to(candidatos_layer)

candidatos_layer.add_to(mapa_inicial)
folium.LayerControl(collapsed=False).add_to(mapa_inicial)

caminho_mapa_inicial = PASTA_SAIDA / "mapa_inicial_candidatos.html"
mapa_inicial.save(str(caminho_mapa_inicial))
print(f"Mapa inicial salvo em: {caminho_mapa_inicial}")
display(mapa_inicial)

Mapa inicial salvo em: saidas_modelo_professor\mapa_inicial_candidatos.html


## 9. Trânsito sintético

Esta célula cria o cenário com trânsito.

O peso original `travel_time` representa o tempo estimado sem congestionamento. Para simular trânsito, criamos `traffic_time`.

Inicialmente:

```text
traffic_time = travel_time
```

Depois, alguns trechos da rota base **A -> B** recebem penalização:

```text
traffic_time = travel_time * fator_de_trânsito
```

O trânsito é sintético, mas útil para comparar como os algoritmos reagem quando algumas vias ficam mais caras.

In [9]:
for _, _, _, dados in G_direcao.edges(keys=True, data=True):
    dados["travel_time"] = float(dados.get("travel_time", dados.get("length", 0) / 13.89))
    dados["traffic_time"] = dados["travel_time"]
    dados["traffic_factor"] = 1.0
    dados["traffic_penalized"] = False

rota_base_A_B = nx.shortest_path(G_direcao, A_direcao, B_direcao, weight="travel_time")
pares_rota_base = list(zip(rota_base_A_B[:-1], rota_base_A_B[1:]))

random.seed(SEMENTE_ALEATORIA)
quantidade_penalizada = max(1, int(len(pares_rota_base) * 0.35))
pares_penalizados = set(random.sample(pares_rota_base, quantidade_penalizada))

for u, v, k, dados in G_direcao.edges(keys=True, data=True):
    if (u, v) in pares_penalizados:
        fator = random.uniform(2.0, 4.0)
        dados["traffic_factor"] = fator
        dados["traffic_time"] = dados["travel_time"] * fator
        dados["traffic_penalized"] = True

arestas_transito = [
    (u, v, k)
    for u, v, k, dados in G_direcao.edges(keys=True, data=True)
    if dados.get("traffic_penalized")
]

print(f"Trechos da rota base A -> B: {len(pares_rota_base)}")
print(f"Arestas penalizadas pelo trânsito sintético: {len(arestas_transito)}")

Trechos da rota base A -> B: 30
Arestas penalizadas pelo trânsito sintético: 10


## 10. Funções auxiliares

Estas funções são usadas por todos os algoritmos.

Elas resolvem tarefas comuns:

- recuperar o menor peso entre duas arestas;
- listar vizinhos com o peso escolhido;
- reconstruir caminho a partir de predecessores;
- transformar uma rota em coordenadas para o mapa;
- calcular o custo total de uma rota.

Separar essas funções evita repetição e deixa os algoritmos mais fáceis de comparar.

In [10]:
def menor_peso_aresta(G, u, v, peso):
    dados_arestas = G.get_edge_data(u, v, default={})
    if not dados_arestas:
        return math.inf
    return min(float(dados.get(peso, dados.get("length", math.inf))) for dados in dados_arestas.values())


def vizinhos_com_peso(G, no, peso):
    for _, vizinho, dados in G.out_edges(no, data=True):
        yield vizinho, float(dados.get(peso, dados.get("length", math.inf)))


def reconstruir_caminho(pred, origem, destino):
    if origem == destino:
        return [origem]
    if destino not in pred:
        return None
    caminho = [destino]
    atual = destino
    while atual != origem:
        atual = pred.get(atual)
        if atual is None:
            return None
        caminho.append(atual)
    caminho.reverse()
    return caminho


def coordenadas_rota(G, rota):
    if not rota:
        return []

    coords = []
    for u, v in zip(rota[:-1], rota[1:]):
        dados_arestas = G.get_edge_data(u, v, default={})
        melhor = None
        if dados_arestas:
            melhor = min(dados_arestas.values(), key=lambda d: float(d.get("length", 0)))

        if melhor and "geometry" in melhor:
            pontos = [(lat, lon) for lon, lat in melhor["geometry"].coords]
        else:
            pontos = [
                (G.nodes[u]["y"], G.nodes[u]["x"]),
                (G.nodes[v]["y"], G.nodes[v]["x"]),
            ]

        if coords and pontos and coords[-1] == pontos[0]:
            coords.extend(pontos[1:])
        else:
            coords.extend(pontos)

    if len(rota) == 1:
        no = rota[0]
        coords.append((G.nodes[no]["y"], G.nodes[no]["x"]))

    return coords


def custo_rota(G, rota, peso):
    if rota is None:
        return math.inf
    total = 0.0
    for u, v in zip(rota[:-1], rota[1:]):
        total += menor_peso_aresta(G, u, v, peso)
    return total

## 11. Dijkstra simples

Implementação base do Dijkstra, sem fila de prioridade.

A cada iteração, o algoritmo escolhe por busca linear o nó aberto com menor custo acumulado.

Regra de relaxamento:

```text
se dist[u] + peso(u, v) < dist[v], atualiza dist[v]
```

Essa versão é usada como baseline de desempenho.

In [11]:
def dijkstra_simples_single_source(G, origem, alvos=None, peso="length"):
    inicio = time.perf_counter()
    alvos = set(alvos) if alvos is not None else None
    dist = {origem: 0.0}
    pred = {}
    visitados = set()
    expandidos = 0

    while True:
        atual = None
        melhor_dist = math.inf
        for no, valor in dist.items():
            if no not in visitados and valor < melhor_dist:
                atual = no
                melhor_dist = valor

        if atual is None:
            break

        visitados.add(atual)
        expandidos += 1

        if alvos is not None and atual in alvos:
            alvos.remove(atual)
            if not alvos:
                break

        for vizinho, peso_aresta in vizinhos_com_peso(G, atual, peso):
            novo_custo = dist[atual] + peso_aresta
            if novo_custo < dist.get(vizinho, math.inf):
                dist[vizinho] = novo_custo
                pred[vizinho] = atual

    return {
        "dist": dist,
        "pred": pred,
        "expandidos": expandidos,
        "tempo_ms": (time.perf_counter() - inicio) * 1000,
    }

## 12. Dijkstra com heap

Esta versão usa fila de prioridade.

A lógica de caminho mínimo é a mesma do Dijkstra simples, mas a escolha do próximo nó é feita com `heapq`.

Em grafos urbanos, que são esparsos, essa estrutura tende a ser mais eficiente.

In [12]:
def dijkstra_heap_single_source(G, origem, alvos=None, peso="length"):
    inicio = time.perf_counter()
    alvos = set(alvos) if alvos is not None else None
    dist = {origem: 0.0}
    pred = {}
    heap = [(0.0, origem)]
    visitados = set()
    expandidos = 0

    while heap:
        custo_atual, atual = heapq.heappop(heap)
        if atual in visitados:
            continue

        visitados.add(atual)
        expandidos += 1

        if alvos is not None and atual in alvos:
            alvos.remove(atual)
            if not alvos:
                break

        for vizinho, peso_aresta in vizinhos_com_peso(G, atual, peso):
            novo_custo = custo_atual + peso_aresta
            if novo_custo < dist.get(vizinho, math.inf):
                dist[vizinho] = novo_custo
                pred[vizinho] = atual
                heapq.heappush(heap, (novo_custo, vizinho))

    return {
        "dist": dist,
        "pred": pred,
        "expandidos": expandidos,
        "tempo_ms": (time.perf_counter() - inicio) * 1000,
    }

## 13. A*

O A* é uma busca informada.

Ele usa:

```text
f(n) = g(n) + h(n)
```

onde:

- `g(n)` é o custo real acumulado;
- `h(n)` é a heurística até o destino.

A heurística usada aqui é a distância em linha reta. Para tempo, essa distância é convertida em segundos usando uma velocidade máxima estimada.

In [13]:
def velocidade_maxima_mps(G):
    valores = []
    for _, _, dados in G.edges(data=True):
        length = float(dados.get("length", 0))
        travel_time = float(dados.get("travel_time", 0))
        if length > 0 and travel_time > 0:
            valores.append(length / travel_time)
    return max(valores) if valores else 13.89


VELOCIDADE_MAXIMA_MPS = velocidade_maxima_mps(G_direcao)


def heuristica_astar(G, no, destino, peso):
    lat1 = G.nodes[no]["y"]
    lon1 = G.nodes[no]["x"]
    lat2 = G.nodes[destino]["y"]
    lon2 = G.nodes[destino]["x"]
    distancia = ox.distance.great_circle(lat1, lon1, lat2, lon2)
    if peso == "length":
        return distancia
    return distancia / VELOCIDADE_MAXIMA_MPS


def astar_ponto_a_ponto(G, origem, destino, peso="length"):
    inicio = time.perf_counter()
    heap = [(heuristica_astar(G, origem, destino, peso), 0.0, origem)]
    dist = {origem: 0.0}
    pred = {}
    visitados = set()
    expandidos = 0

    while heap:
        _, custo_atual, atual = heapq.heappop(heap)
        if atual in visitados:
            continue

        visitados.add(atual)
        expandidos += 1

        if atual == destino:
            return {
                "caminho": reconstruir_caminho(pred, origem, destino),
                "custo": custo_atual,
                "expandidos": expandidos,
                "tempo_ms": (time.perf_counter() - inicio) * 1000,
            }

        for vizinho, peso_aresta in vizinhos_com_peso(G, atual, peso):
            novo_custo = custo_atual + peso_aresta
            if novo_custo < dist.get(vizinho, math.inf):
                dist[vizinho] = novo_custo
                pred[vizinho] = atual
                prioridade = novo_custo + heuristica_astar(G, vizinho, destino, peso)
                heapq.heappush(heap, (prioridade, novo_custo, vizinho))

    return {
        "caminho": None,
        "custo": math.inf,
        "expandidos": expandidos,
        "tempo_ms": (time.perf_counter() - inicio) * 1000,
    }

## 14. Dijkstra bidirecional

O Dijkstra bidirecional executa duas buscas:

- uma a partir da origem;
- outra a partir do destino, no grafo reverso.

Quando as buscas se encontram, o caminho é reconstruído.

Ele é usado como quarto algoritmo comparativo, representando uma abordagem eficiente para rotas ponto a ponto.

In [14]:
def dijkstra_bidirecional_ponto_a_ponto(G, origem, destino, peso="length"):
    inicio = time.perf_counter()
    if origem == destino:
        return {"caminho": [origem], "custo": 0.0, "expandidos": 1, "tempo_ms": 0.0}

    G_rev = G.reverse(copy=False)
    dist_f = {origem: 0.0}
    dist_b = {destino: 0.0}
    pred_f = {}
    pred_b = {}
    heap_f = [(0.0, origem)]
    heap_b = [(0.0, destino)]
    fechados_f = set()
    fechados_b = set()
    melhor = math.inf
    encontro = None
    expandidos = 0

    while heap_f and heap_b:
        if heap_f[0][0] + heap_b[0][0] >= melhor:
            break

        if heap_f[0][0] <= heap_b[0][0]:
            custo_atual, atual = heapq.heappop(heap_f)
            if atual in fechados_f:
                continue
            fechados_f.add(atual)
            expandidos += 1

            if atual in fechados_b:
                total = dist_f[atual] + dist_b[atual]
                if total < melhor:
                    melhor = total
                    encontro = atual

            for vizinho, peso_aresta in vizinhos_com_peso(G, atual, peso):
                novo_custo = custo_atual + peso_aresta
                if novo_custo < dist_f.get(vizinho, math.inf):
                    dist_f[vizinho] = novo_custo
                    pred_f[vizinho] = atual
                    heapq.heappush(heap_f, (novo_custo, vizinho))
        else:
            custo_atual, atual = heapq.heappop(heap_b)
            if atual in fechados_b:
                continue
            fechados_b.add(atual)
            expandidos += 1

            if atual in fechados_f:
                total = dist_f[atual] + dist_b[atual]
                if total < melhor:
                    melhor = total
                    encontro = atual

            for vizinho, peso_aresta in vizinhos_com_peso(G_rev, atual, peso):
                novo_custo = custo_atual + peso_aresta
                if novo_custo < dist_b.get(vizinho, math.inf):
                    dist_b[vizinho] = novo_custo
                    pred_b[vizinho] = atual
                    heapq.heappush(heap_b, (novo_custo, vizinho))

    if encontro is None:
        intersecao = set(dist_f) & set(dist_b)
        if intersecao:
            encontro = min(intersecao, key=lambda n: dist_f[n] + dist_b[n])
            melhor = dist_f[encontro] + dist_b[encontro]

    if encontro is None:
        return {
            "caminho": None,
            "custo": math.inf,
            "expandidos": expandidos,
            "tempo_ms": (time.perf_counter() - inicio) * 1000,
        }

    caminho_f = reconstruir_caminho(pred_f, origem, encontro)
    caminho_b = reconstruir_caminho(pred_b, destino, encontro)
    if caminho_f is None or caminho_b is None:
        return {
            "caminho": None,
            "custo": math.inf,
            "expandidos": expandidos,
            "tempo_ms": (time.perf_counter() - inicio) * 1000,
        }

    caminho = caminho_f + list(reversed(caminho_b))[1:]

    return {
        "caminho": caminho,
        "custo": melhor,
        "expandidos": expandidos,
        "tempo_ms": (time.perf_counter() - inicio) * 1000,
    }

## 15. Funções de avaliação

Esta célula implementa a função de custo do modelo do professor.

Para cada candidato **P_i**, calculamos:

```text
D_walk(A, P_i)
T_walk(A, P_i)
D_drive(P_i, B)
T_drive(P_i, B)
```

Para distância:

```text
Custo_D(P_i) = D_walk(A, P_i) + D_drive(P_i, B)
```

Para tempo:

```text
Custo_T(P_i) = T_walk(A, P_i) + T_drive(P_i, B)
```

O caso sem caminhada é representado por:

```text
P = A
```

Assim, ele serve como base para medir se caminhar trouxe ganho.

In [15]:
def montar_linha_resultado(nome_algoritmo, metrica, candidato, rota_P_B, expandidos, tempo_execucao_ms, peso):
    d_walk = float(candidato["distancia_total_a_pe_m"])
    t_walk = float(candidato["tempo_caminhada_s"])
    d_drive = custo_rota(G_direcao, rota_P_B, "length")
    peso_tempo = peso if metrica == "tempo" else "traffic_time"
    t_drive = custo_rota(G_direcao, rota_P_B, peso_tempo)

    if metrica == "distancia":
        custo = d_walk + d_drive
    else:
        custo = t_walk + t_drive

    return {
        "algoritmo": nome_algoritmo,
        "metrica": metrica,
        "P": candidato["id"],
        "no_caminhada": int(candidato["no_caminhada"]),
        "no_direcao": int(candidato["no_direcao"]),
        "latitude_pedestre": float(candidato["latitude_pedestre"]),
        "longitude_pedestre": float(candidato["longitude_pedestre"]),
        "latitude_embarque": float(candidato["latitude_embarque"]),
        "longitude_embarque": float(candidato["longitude_embarque"]),
        "D_walk_m": d_walk,
        "T_walk_s": t_walk,
        "D_drive_P_B_m": d_drive,
        "T_drive_P_B_s": t_drive,
        "custo": custo,
        "rota_P_B": rota_P_B,
        "expandidos": expandidos,
        "tempo_execucao_ms": tempo_execucao_ms,
    }


def avaliar_single_source(nome_algoritmo, funcao, peso, metrica):
    inicio = time.perf_counter()
    alvos = {int(no) for no in df_candidatos_embarque_P["no_direcao"].tolist()}
    reverso = G_direcao.reverse(copy=False)

    resultado_B_reverso = funcao(reverso, B_direcao, alvos=alvos, peso=peso)

    linhas = []
    for _, candidato in df_candidatos_embarque_P.iterrows():
        no_p = int(candidato["no_direcao"])
        if no_p not in resultado_B_reverso["dist"]:
            continue

        rota_B_P_reversa = reconstruir_caminho(resultado_B_reverso["pred"], B_direcao, no_p)
        if rota_B_P_reversa is None:
            continue

        rota_P_B = list(reversed(rota_B_P_reversa))
        linhas.append(
            montar_linha_resultado(
                nome_algoritmo,
                metrica,
                candidato,
                rota_P_B,
                resultado_B_reverso["expandidos"],
                resultado_B_reverso["tempo_ms"],
                peso,
            )
        )

    df = pd.DataFrame(linhas)
    if df.empty:
        raise ValueError(f"Nenhum candidato P gerou rota válida para {nome_algoritmo} com peso {peso}.")

    melhor = df.sort_values("custo").iloc[0].to_dict()
    melhor["tempo_total_celula_ms"] = (time.perf_counter() - inicio) * 1000
    return df, melhor


def avaliar_ponto_a_ponto(nome_algoritmo, funcao, peso, metrica):
    inicio = time.perf_counter()
    linhas = []
    expandidos_total = 0
    tempo_exec_total = 0.0

    for _, candidato in df_candidatos_embarque_P.iterrows():
        no_p = int(candidato["no_direcao"])
        resultado = funcao(G_direcao, no_p, B_direcao, peso=peso)
        expandidos_total += resultado["expandidos"]
        tempo_exec_total += resultado["tempo_ms"]

        if resultado["caminho"] is None:
            continue

        linhas.append(
            montar_linha_resultado(
                nome_algoritmo,
                metrica,
                candidato,
                resultado["caminho"],
                resultado["expandidos"],
                resultado["tempo_ms"],
                peso,
            )
        )

    df = pd.DataFrame(linhas)
    if df.empty:
        raise ValueError(f"Nenhum candidato P gerou rota válida para {nome_algoritmo} com peso {peso}.")

    melhor = df.sort_values("custo").iloc[0].to_dict()
    melhor["expandidos"] = expandidos_total
    melhor["tempo_execucao_ms"] = tempo_exec_total
    melhor["tempo_total_celula_ms"] = (time.perf_counter() - inicio) * 1000
    return df, melhor


def montar_resultado_sem_caminhada(nome_algoritmo, metrica, peso, rota_A_B, expandidos, tempo_execucao_ms):
    d_drive = custo_rota(G_direcao, rota_A_B, "length")
    peso_tempo = peso if metrica == "tempo" else "traffic_time"
    t_drive = custo_rota(G_direcao, rota_A_B, peso_tempo)
    custo = d_drive if metrica == "distancia" else t_drive

    return {
        "algoritmo": nome_algoritmo,
        "metrica": metrica,
        "P": "A",
        "no_caminhada": A_caminhada,
        "no_direcao": A_direcao,
        "latitude_pedestre": lat_A,
        "longitude_pedestre": lon_A,
        "latitude_embarque": G_direcao.nodes[A_direcao]["y"],
        "longitude_embarque": G_direcao.nodes[A_direcao]["x"],
        "D_walk_m": 0.0,
        "T_walk_s": 0.0,
        "D_drive_P_B_m": d_drive,
        "T_drive_P_B_s": t_drive,
        "custo": custo,
        "rota_P_B": rota_A_B,
        "expandidos": expandidos,
        "tempo_execucao_ms": tempo_execucao_ms,
    }


def avaliar_sem_caminhada_single_source(nome_algoritmo, funcao, peso, metrica):
    resultado = funcao(G_direcao, A_direcao, alvos={B_direcao}, peso=peso)
    rota_A_B = reconstruir_caminho(resultado["pred"], A_direcao, B_direcao)
    if rota_A_B is None:
        raise ValueError("Não foi possível calcular o caso sem caminhada.")
    return montar_resultado_sem_caminhada(
        nome_algoritmo,
        metrica,
        peso,
        rota_A_B,
        resultado["expandidos"],
        resultado["tempo_ms"],
    )


def avaliar_sem_caminhada_ponto_a_ponto(nome_algoritmo, funcao, peso, metrica):
    resultado = funcao(G_direcao, A_direcao, B_direcao, peso=peso)
    if resultado["caminho"] is None:
        raise ValueError("Não foi possível calcular o caso sem caminhada.")
    return montar_resultado_sem_caminhada(
        nome_algoritmo,
        metrica,
        peso,
        resultado["caminho"],
        resultado["expandidos"],
        resultado["tempo_ms"],
    )

## 16. Função de mapa

Esta célula desenha soluções no mapa.

Para cada resultado, o mapa mostra:

- trecho a pé **A -> P** em verde;
- conexão curta entre o ponto caminhável e o ponto de embarque na malha de carro, quando existir;
- ponto **P** do pedestre;
- ícone de carro no ponto exato de embarque;
- trecho de carro **P -> B**.

O caso sem caminhada aparece como **P = A**.

In [16]:
def adicionar_solucao_mapa(mapa, nome_layer, resultado, cor_carro, cor_pedestre, show=False):
    layer = folium.FeatureGroup(name=nome_layer, show=show)

    if resultado["P"] != "A":
        try:
            rota_a_p = nx.shortest_path(
                G_caminhada,
                A_caminhada,
                int(resultado["no_caminhada"]),
                weight="length",
            )
            folium.PolyLine(
                coordenadas_rota(G_caminhada, rota_a_p),
                color=cor_pedestre,
                weight=4,
                opacity=0.9,
                tooltip=f"{nome_layer} - caminhada A até P",
            ).add_to(layer)
        except Exception:
            folium.PolyLine(
                [(lat_A, lon_A), (resultado["latitude_pedestre"], resultado["longitude_pedestre"])],
                color=cor_pedestre,
                weight=4,
                opacity=0.7,
                tooltip=f"{nome_layer} - caminhada aproximada",
            ).add_to(layer)

    folium.PolyLine(
        [
            (resultado["latitude_pedestre"], resultado["longitude_pedestre"]),
            (resultado["latitude_embarque"], resultado["longitude_embarque"]),
        ],
        color="#f59e0b",
        weight=3,
        opacity=0.8,
        dash_array="4, 6",
        tooltip=f"{nome_layer} - acesso ao embarque",
    ).add_to(layer)

    folium.PolyLine(
        coordenadas_rota(G_direcao, resultado["rota_P_B"]),
        color=cor_carro,
        weight=5,
        opacity=0.82,
        tooltip=f"{nome_layer} - carro P até B",
    ).add_to(layer)

    folium.CircleMarker(
        [resultado["latitude_pedestre"], resultado["longitude_pedestre"]],
        radius=8 if resultado["P"] != "A" else 6,
        color=cor_pedestre,
        fill=True,
        fill_color=cor_pedestre,
        fill_opacity=0.95,
        tooltip=f"P do pedestre: {resultado['P']}",
        popup=f"<b>{nome_layer}</b><br>P: {resultado['P']}<br>Custo: {resultado['custo']:.2f}",
    ).add_to(layer)

    folium.Marker(
        [resultado["latitude_embarque"], resultado["longitude_embarque"]],
        tooltip=f"Embarque de carro: {resultado['P']}",
        popup=f"Embarque na malha de carro para {resultado['P']}",
        icon=folium.Icon(color="blue", icon="car", prefix="fa"),
    ).add_to(layer)

    layer.add_to(mapa)
    return layer


def montar_mapa_consolidado(solucoes):
    mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=13, tiles="CartoDB positron")
    grupos = {"Base": [], "Trânsito sintético": []}

    base = folium.FeatureGroup(name="Base | A, B, raio e candidatos", show=True)
    folium.Marker([lat_A, lon_A], popup=ROTULO_A, tooltip="A - passageiro", icon=folium.Icon(color="green", icon="user")).add_to(base)
    folium.Marker([lat_B, lon_B], popup=ROTULO_B, tooltip="B - destino", icon=folium.Icon(color="red", icon="flag")).add_to(base)
    folium.Circle([lat_A, lon_A], radius=RAIO_CAMINHADA_M, color="#16a34a", fill=False, weight=3, tooltip=f"Raio X: {RAIO_CAMINHADA_M} m").add_to(base)

    for _, row in df_candidatos_P.iterrows():
        cor = "#374151" if row["valido_para_embarque"] else "#9ca3af"
        folium.CircleMarker(
            [row["latitude_pedestre"], row["longitude_pedestre"]],
            radius=3 if row["valido_para_embarque"] else 2,
            color=cor,
            fill=True,
            fill_color=cor,
            fill_opacity=0.6,
            tooltip=f"{row['id']} - candidato P",
        ).add_to(base)

    base.add_to(mapa)
    grupos["Base"].append(base)

    transito = folium.FeatureGroup(name="Trânsito sintético | vias penalizadas", show=False)
    for u, v, k, dados in G_direcao.edges(keys=True, data=True):
        if dados.get("traffic_penalized", False):
            folium.PolyLine(
                coordenadas_rota(G_direcao, [u, v]),
                color="#dc2626",
                weight=4,
                opacity=0.72,
                tooltip=f"Via penalizada - fator {dados.get('traffic_factor', 1):.2f}",
            ).add_to(transito)
    transito.add_to(mapa)
    grupos["Trânsito sintético"].append(transito)

    cores = {
        "Dijkstra simples": ("#ef4444", "#16a34a"),
        "Dijkstra com heap": ("#2563eb", "#22c55e"),
        "A*": ("#7c3aed", "#84cc16"),
        "Dijkstra bidirecional": ("#0891b2", "#10b981"),
    }

    for grupo_nome, itens in solucoes.items():
        grupos[grupo_nome] = []
        for algoritmo, criterio, resultado, show in itens:
            cor_carro, cor_pedestre = cores[algoritmo]
            nome_layer = f"{algoritmo} | {criterio}"
            layer = adicionar_solucao_mapa(
                mapa,
                nome_layer,
                resultado,
                "#475569" if "ponto A" in criterio else cor_carro,
                cor_pedestre,
                show=show,
            )
            grupos[grupo_nome].append(layer)

    try:
        plugins.GroupedLayerControl(groups=grupos, collapsed=False, exclusive_groups=False).add_to(mapa)
    except Exception:
        folium.LayerControl(collapsed=False).add_to(mapa)

    caminho = PASTA_SAIDA / "mapa_consolidado_modelo_professor.html"
    mapa.save(str(caminho))
    print(f"Mapa consolidado salvo em: {caminho}")
    display(mapa)
    return mapa

## 17. Execução do Dijkstra simples

Esta célula executa o primeiro algoritmo comparado: o **Dijkstra simples**, implementado sem heap.

O objetivo aqui é usar essa versão como linha de base. Para cada critério, o algoritmo calcula:

- o melhor ponto **P** considerando caminhada;
- o caso sem caminhada, isto é, **A = P**.

São avaliados três pesos diferentes no grafo de carros:

- `length`: menor rota em distância;
- `travel_time`: rota mais rápida sem trânsito;
- `traffic_time`: rota mais rápida com trânsito sintético.

Como essa versão é `single-source`, o algoritmo parte do nó de embarque e calcula custos para os alvos necessários. No final, a tabela mostra lado a lado os melhores resultados com caminhada e os resultados equivalentes sem caminhada. Isso permite comparar custo, ponto escolhido, nós expandidos e tempo de execução.

In [17]:
df_dijkstra_simples_distancia, melhor_dijkstra_simples_distancia = avaliar_single_source("Dijkstra simples", dijkstra_simples_single_source, "length", "distancia")
sem_dijkstra_simples_distancia = avaliar_sem_caminhada_single_source("Dijkstra simples", dijkstra_simples_single_source, "length", "distancia")

df_dijkstra_simples_tempo_sem, melhor_dijkstra_simples_tempo_sem = avaliar_single_source("Dijkstra simples", dijkstra_simples_single_source, "travel_time", "tempo")
sem_dijkstra_simples_tempo_sem = avaliar_sem_caminhada_single_source("Dijkstra simples", dijkstra_simples_single_source, "travel_time", "tempo")

df_dijkstra_simples_tempo_com, melhor_dijkstra_simples_tempo_com = avaliar_single_source("Dijkstra simples", dijkstra_simples_single_source, "traffic_time", "tempo")
sem_dijkstra_simples_tempo_com = avaliar_sem_caminhada_single_source("Dijkstra simples", dijkstra_simples_single_source, "traffic_time", "tempo")

pd.DataFrame([
    melhor_dijkstra_simples_distancia,
    sem_dijkstra_simples_distancia,
    melhor_dijkstra_simples_tempo_sem,
    sem_dijkstra_simples_tempo_sem,
    melhor_dijkstra_simples_tempo_com,
    sem_dijkstra_simples_tempo_com,
]).drop(columns=["rota_P_B"])

,algoritmo,metrica,P,no_caminhada,no_direcao,latitude_pedestre,longitude_pedestre,latitude_embarque,longitude_embarque,D_walk_m,T_walk_s,D_drive_P_B_m,T_drive_P_B_s,custo,expandidos,tempo_execucao_ms,tempo_total_celula_ms
0,Dijkstra simples,distancia,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3453.373452,295.444346,3679.883452,2582,200.8503,212.7287
1,Dijkstra simples,distancia,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,269.545455,3307.552069,3109,304.6506,NaN
2,Dijkstra simples,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3456.777255,173.259790,336.349790,1776,111.3204,119.3768
3,Dijkstra simples,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,165.239820,165.239820,1766,123.6080,NaN
4,Dijkstra simples,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3462.172548,206.260458,369.350458,2194,175.6984,184.8401
5,Dijkstra simples,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3312.947362,199.699267,199.699267,2347,186.4226,NaN


## 18. Execução do Dijkstra com heap

Esta célula repete a mesma avaliação usando **Dijkstra com heap**, ou seja, com fila de prioridade.

A lógica matemática continua sendo a de menor caminho, mas a estrutura de dados muda. Em vez de procurar linearmente o próximo nó de menor custo, o algoritmo usa `heapq` para retirar o melhor candidato com menor custo acumulado.

Assim como no Dijkstra simples, esta célula roda seis casos:

- com caminhada, escolhendo o melhor **P** por distância;
- sem caminhada, usando **A = P**, por distância;
- com caminhada, escolhendo o melhor **P** por tempo sem trânsito;
- sem caminhada, usando **A = P**, por tempo sem trânsito;
- com caminhada, escolhendo o melhor **P** por tempo com trânsito sintético;
- sem caminhada, usando **A = P**, por tempo com trânsito sintético.

A comparação com a célula anterior ajuda a mostrar o impacto de uma estrutura de dados mais eficiente sobre o mesmo problema de roteamento.

In [18]:
df_dijkstra_heap_distancia, melhor_dijkstra_heap_distancia = avaliar_single_source("Dijkstra com heap", dijkstra_heap_single_source, "length", "distancia")
sem_dijkstra_heap_distancia = avaliar_sem_caminhada_single_source("Dijkstra com heap", dijkstra_heap_single_source, "length", "distancia")

df_dijkstra_heap_tempo_sem, melhor_dijkstra_heap_tempo_sem = avaliar_single_source("Dijkstra com heap", dijkstra_heap_single_source, "travel_time", "tempo")
sem_dijkstra_heap_tempo_sem = avaliar_sem_caminhada_single_source("Dijkstra com heap", dijkstra_heap_single_source, "travel_time", "tempo")

df_dijkstra_heap_tempo_com, melhor_dijkstra_heap_tempo_com = avaliar_single_source("Dijkstra com heap", dijkstra_heap_single_source, "traffic_time", "tempo")
sem_dijkstra_heap_tempo_com = avaliar_sem_caminhada_single_source("Dijkstra com heap", dijkstra_heap_single_source, "traffic_time", "tempo")

pd.DataFrame([
    melhor_dijkstra_heap_distancia,
    sem_dijkstra_heap_distancia,
    melhor_dijkstra_heap_tempo_sem,
    sem_dijkstra_heap_tempo_sem,
    melhor_dijkstra_heap_tempo_com,
    sem_dijkstra_heap_tempo_com,
]).drop(columns=["rota_P_B"])

,algoritmo,metrica,P,no_caminhada,no_direcao,latitude_pedestre,longitude_pedestre,latitude_embarque,longitude_embarque,D_walk_m,T_walk_s,D_drive_P_B_m,T_drive_P_B_s,custo,expandidos,tempo_execucao_ms,tempo_total_celula_ms
0,Dijkstra com heap,distancia,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3453.373452,295.444346,3679.883452,2582,21.3296,29.7649
1,Dijkstra com heap,distancia,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,269.545455,3307.552069,3109,23.8005,NaN
2,Dijkstra com heap,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3456.777255,173.259790,336.349790,1776,14.8961,22.3190
3,Dijkstra com heap,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,165.239820,165.239820,1766,13.4180,NaN
4,Dijkstra com heap,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3462.172548,206.260458,369.350458,2194,18.1943,25.9223
5,Dijkstra com heap,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3312.947362,199.699267,199.699267,2347,17.7590,NaN


## 19. Execução do A*

Esta célula executa o **A*** para os mesmos critérios.

Ao contrário dos Dijkstras `single-source`, o A* é uma busca ponto a ponto. Por isso, para avaliar candidatos **P**, o notebook executa o algoritmo individualmente para cada ponto candidato válido e depois escolhe aquele com menor custo total.

O A* usa a função:

```text
f(n) = g(n) + h(n)
```

onde:

- `g(n)` é o custo real já acumulado no grafo;
- `h(n)` é uma estimativa do custo até o destino, calculada por distância geográfica em linha reta.

Nesta célula, o A* também é executado para distância, tempo sem trânsito e tempo com trânsito sintético, sempre comparando a solução com caminhada contra o caso sem caminhada.

In [19]:
df_astar_distancia, melhor_astar_distancia = avaliar_ponto_a_ponto("A*", astar_ponto_a_ponto, "length", "distancia")
sem_astar_distancia = avaliar_sem_caminhada_ponto_a_ponto("A*", astar_ponto_a_ponto, "length", "distancia")

df_astar_tempo_sem, melhor_astar_tempo_sem = avaliar_ponto_a_ponto("A*", astar_ponto_a_ponto, "travel_time", "tempo")
sem_astar_tempo_sem = avaliar_sem_caminhada_ponto_a_ponto("A*", astar_ponto_a_ponto, "travel_time", "tempo")

df_astar_tempo_com, melhor_astar_tempo_com = avaliar_ponto_a_ponto("A*", astar_ponto_a_ponto, "traffic_time", "tempo")
sem_astar_tempo_com = avaliar_sem_caminhada_ponto_a_ponto("A*", astar_ponto_a_ponto, "traffic_time", "tempo")

pd.DataFrame([
    melhor_astar_distancia,
    sem_astar_distancia,
    melhor_astar_tempo_sem,
    sem_astar_tempo_sem,
    melhor_astar_tempo_com,
    sem_astar_tempo_com,
]).drop(columns=["rota_P_B"])

,algoritmo,metrica,P,no_caminhada,no_direcao,latitude_pedestre,longitude_pedestre,latitude_embarque,longitude_embarque,D_walk_m,T_walk_s,D_drive_P_B_m,T_drive_P_B_s,custo,expandidos,tempo_execucao_ms,tempo_total_celula_ms
0,A*,distancia,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3453.373452,295.444346,3679.883452,3510,57.112100,66.3466
1,A*,distancia,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,269.545455,3307.552069,68,1.132200,NaN
2,A*,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3456.777255,173.259790,336.349790,4499,72.769299,84.1839
3,A*,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,165.239820,165.239820,64,1.131400,NaN
4,A*,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3462.172548,206.260458,369.350458,13564,221.641000,236.1230
5,A*,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3312.947362,199.699267,199.699267,233,4.921300,NaN


## 20. Execução do Dijkstra bidirecional

Esta célula executa o **Dijkstra bidirecional**.

A ideia desse algoritmo é reduzir o espaço de busca fazendo duas expansões simultâneas:

- uma busca sai da origem;
- outra busca sai do destino, usando o grafo reverso;
- quando as duas buscas se encontram, o caminho é reconstruído.

Como ele também é naturalmente ponto a ponto, o notebook avalia cada candidato **P** separadamente. Para cada candidato, calcula a rota de carro de **P** até **B** com o peso escolhido.

Assim como nas outras células de execução, são comparados:

- menor distância;
- menor tempo sem trânsito;
- menor tempo com trânsito sintético;
- a versão equivalente sem caminhada, usando **A = P**.

O resultado permite observar se a busca bidirecional encontra a mesma solução de menor custo e como ela se comporta em quantidade de nós expandidos e tempo de processamento.

In [20]:
df_bidirecional_distancia, melhor_bidirecional_distancia = avaliar_ponto_a_ponto("Dijkstra bidirecional", dijkstra_bidirecional_ponto_a_ponto, "length", "distancia")
sem_bidirecional_distancia = avaliar_sem_caminhada_ponto_a_ponto("Dijkstra bidirecional", dijkstra_bidirecional_ponto_a_ponto, "length", "distancia")

df_bidirecional_tempo_sem, melhor_bidirecional_tempo_sem = avaliar_ponto_a_ponto("Dijkstra bidirecional", dijkstra_bidirecional_ponto_a_ponto, "travel_time", "tempo")
sem_bidirecional_tempo_sem = avaliar_sem_caminhada_ponto_a_ponto("Dijkstra bidirecional", dijkstra_bidirecional_ponto_a_ponto, "travel_time", "tempo")

df_bidirecional_tempo_com, melhor_bidirecional_tempo_com = avaliar_ponto_a_ponto("Dijkstra bidirecional", dijkstra_bidirecional_ponto_a_ponto, "traffic_time", "tempo")
sem_bidirecional_tempo_com = avaliar_sem_caminhada_ponto_a_ponto("Dijkstra bidirecional", dijkstra_bidirecional_ponto_a_ponto, "traffic_time", "tempo")

pd.DataFrame([
    melhor_bidirecional_distancia,
    sem_bidirecional_distancia,
    melhor_bidirecional_tempo_sem,
    sem_bidirecional_tempo_sem,
    melhor_bidirecional_tempo_com,
    sem_bidirecional_tempo_com,
]).drop(columns=["rota_P_B"])

,algoritmo,metrica,P,no_caminhada,no_direcao,latitude_pedestre,longitude_pedestre,latitude_embarque,longitude_embarque,D_walk_m,T_walk_s,D_drive_P_B_m,T_drive_P_B_s,custo,expandidos,tempo_execucao_ms,tempo_total_celula_ms
0,Dijkstra bidirecional,distancia,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3453.373452,295.444346,3679.883452,64299,518.4411,541.3480
1,Dijkstra bidirecional,distancia,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,269.545455,3307.552069,1461,11.2776,NaN
2,Dijkstra bidirecional,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3456.777255,173.259790,336.349790,27659,204.8361,221.8059
3,Dijkstra bidirecional,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3307.552069,165.239820,165.239820,566,4.0417,NaN
4,Dijkstra bidirecional,tempo,P18,9516583709,505552959,-5.812144,-35.206590,-5.812324,-35.206566,226.51,163.09,3462.172548,206.260458,369.350458,37441,278.9119,298.0521
5,Dijkstra bidirecional,tempo,A,567705822,501969167,-5.812120,-35.205724,-5.812700,-35.205126,0.00,0.00,3312.947362,199.699267,199.699267,813,6.5256,NaN


## 21. Consolidação e ganho

Esta célula organiza os resultados em uma tabela única.

Até aqui, cada algoritmo produziu resultados separados para cada critério. Esta etapa junta tudo em uma única estrutura para facilitar a comparação.

O ganho é calculado sempre em relação ao caso sem caminhada:

```text
ganho = custo_sem_caminhada - custo_com_melhor_P
```

Se o ganho for positivo, caminhar ajudou. Se for negativo, o caso sem caminhada é melhor.

Também são preservadas métricas importantes para análise de desempenho:

- número de nós expandidos;
- tempo de execução em milissegundos;
- distância caminhando;
- tempo caminhando;
- custo da rota de carro.

A tabela gerada nesta célula é salva em CSV para que os resultados possam ser usados no relatório ou em gráficos externos.

In [21]:
def resumir_resultado(nome_algoritmo, criterio, melhor, sem_caminhada):
    custo_com_p = float(melhor["custo"])
    custo_sem = float(sem_caminhada["custo"])
    ganho = custo_sem - custo_com_p
    ganho_percentual = (100 * ganho / custo_sem) if custo_sem else 0.0
    unidade = "m" if "distância" in criterio else "s"
    return {
        "algoritmo": nome_algoritmo,
        "criterio": criterio,
        "unidade": unidade,
        "P_escolhido": str(melhor["P"]),
        "custo_com_P": custo_com_p,
        "custo_sem_caminhada": custo_sem,
        "ganho": ganho,
        "ganho_percentual": ganho_percentual,
        "caminhar_ajudou": bool(ganho > 0),
        "D_walk_m": float(melhor.get("D_walk_m", 0.0)),
        "T_walk_min": float(melhor.get("T_walk_s", 0.0)) / 60,
        "D_drive_P_B_m": float(melhor.get("D_drive_P_B_m", 0.0)),
        "T_drive_P_B_min": float(melhor.get("T_drive_P_B_s", 0.0)) / 60,
        "nos_expandidos_com_P": int(melhor.get("expandidos", 0)),
        "nos_expandidos_sem_caminhada": int(sem_caminhada.get("expandidos", 0)),
        "tempo_execucao_ms_com_P": float(melhor.get("tempo_execucao_ms", 0.0)),
        "tempo_execucao_ms_sem_caminhada": float(sem_caminhada.get("tempo_execucao_ms", 0.0)),
    }


pares_resultados = [
    ("Dijkstra simples", "menor rota em distância (do ponto P)", melhor_dijkstra_simples_distancia, sem_dijkstra_simples_distancia),
    ("Dijkstra simples", "rota mais rápida sem trânsito (do ponto P)", melhor_dijkstra_simples_tempo_sem, sem_dijkstra_simples_tempo_sem),
    ("Dijkstra simples", "rota mais rápida com trânsito sintético (do ponto P)", melhor_dijkstra_simples_tempo_com, sem_dijkstra_simples_tempo_com),
    ("Dijkstra com heap", "menor rota em distância (do ponto P)", melhor_dijkstra_heap_distancia, sem_dijkstra_heap_distancia),
    ("Dijkstra com heap", "rota mais rápida sem trânsito (do ponto P)", melhor_dijkstra_heap_tempo_sem, sem_dijkstra_heap_tempo_sem),
    ("Dijkstra com heap", "rota mais rápida com trânsito sintético (do ponto P)", melhor_dijkstra_heap_tempo_com, sem_dijkstra_heap_tempo_com),
    ("A*", "menor rota em distância (do ponto P)", melhor_astar_distancia, sem_astar_distancia),
    ("A*", "rota mais rápida sem trânsito (do ponto P)", melhor_astar_tempo_sem, sem_astar_tempo_sem),
    ("A*", "rota mais rápida com trânsito sintético (do ponto P)", melhor_astar_tempo_com, sem_astar_tempo_com),
    ("Dijkstra bidirecional", "menor rota em distância (do ponto P)", melhor_bidirecional_distancia, sem_bidirecional_distancia),
    ("Dijkstra bidirecional", "rota mais rápida sem trânsito (do ponto P)", melhor_bidirecional_tempo_sem, sem_bidirecional_tempo_sem),
    ("Dijkstra bidirecional", "rota mais rápida com trânsito sintético (do ponto P)", melhor_bidirecional_tempo_com, sem_bidirecional_tempo_com),
]

df_resultados_consolidados = pd.DataFrame([
    resumir_resultado(nome, criterio, melhor, sem_caminhada)
    for nome, criterio, melhor, sem_caminhada in pares_resultados
])

caminho_resultados = PASTA_SAIDA / "resultados_consolidados_modelo_professor.csv"
df_resultados_consolidados.to_csv(caminho_resultados, index=False, encoding="utf-8-sig")

print(f"Tabela consolidada salva em: {caminho_resultados}")
display(df_resultados_consolidados)

Tabela consolidada salva em: saidas_modelo_professor\resultados_consolidados_modelo_professor.csv


,algoritmo,criterio,unidade,P_escolhido,custo_com_P,custo_sem_caminhada,ganho,ganho_percentual,caminhar_ajudou,D_walk_m,T_walk_min,D_drive_P_B_m,T_drive_P_B_min,nos_expandidos_com_P,nos_expandidos_sem_caminhada,tempo_execucao_ms_com_P,tempo_execucao_ms_sem_caminhada
0,Dijkstra simples,menor rota em distância (do ponto P),m,P18,3679.883452,3307.552069,-372.331384,-11.257007,False,226.51,2.718167,3453.373452,4.924072,2582,3109,200.850300,304.6506
1,Dijkstra simples,rota mais rápida sem trânsito (do ponto P),s,P18,336.349790,165.239820,-171.109970,-103.552503,False,226.51,2.718167,3456.777255,2.887663,1776,1766,111.320400,123.6080
2,Dijkstra simples,rota mais rápida com trânsito sintético (do po...,s,P18,369.350458,199.699267,-169.651191,-84.953337,False,226.51,2.718167,3462.172548,3.437674,2194,2347,175.698400,186.4226
3,Dijkstra com heap,menor rota em distância (do ponto P),m,P18,3679.883452,3307.552069,-372.331384,-11.257007,False,226.51,2.718167,3453.373452,4.924072,2582,3109,21.329600,23.8005
4,Dijkstra com heap,rota mais rápida sem trânsito (do ponto P),s,P18,336.349790,165.239820,-171.109970,-103.552503,False,226.51,2.718167,3456.777255,2.887663,1776,1766,14.896100,13.4180
5,Dijkstra com heap,rota mais rápida com trânsito sintético (do po...,s,P18,369.350458,199.699267,-169.651191,-84.953337,False,226.51,2.718167,3462.172548,3.437674,2194,2347,18.194300,17.7590
6,A*,menor rota em distância (do ponto P),m,P18,3679.883452,3307.552069,-372.331384,-11.257007,False,226.51,2.718167,3453.373452,4.924072,3510,68,57.112100,1.1322
7,A*,rota mais rápida sem trânsito (do ponto P),s,P18,336.349790,165.239820,-171.109970,-103.552503,False,226.51,2.718167,3456.777255,2.887663,4499,64,72.769299,1.1314
8,A*,rota mais rápida com trânsito sintético (do po...,s,P18,369.350458,199.699267,-169.651191,-84.953337,False,226.51,2.718167,3462.172548,3.437674,13564,233,221.641000,4.9213
9,Dijkstra bidirecional,menor rota em distância (do ponto P),m,P18,3679.883452,3307.552069,-372.331384,-11.257007,False,226.51,2.718167,3453.373452,4.924072,64299,1461,518.441100,11.2776


## 22. Ganho ao caminhar

Esta célula mostra apenas as informações necessárias para responder:

```text
Valeu a pena caminhar até o ponto P?
```

O cálculo usado é:

```text
ganho = custo_sem_caminhada - custo_com_P
```

Interpretação:

- `ganho > 0`: caminhar ajudou;
- `ganho = 0`: caminhar não mudou o resultado;
- `ganho < 0`: caminhar piorou, então o caso sem caminhada é melhor.

A unidade depende do critério:

- metros para menor rota em distância;
- segundos para rotas mais rápidas.

Esta tabela é uma versão reduzida da consolidação anterior. Ela foi separada para deixar a interpretação do ganho mais direta durante a apresentação.

In [22]:
df_ganho_ao_caminhar = df_resultados_consolidados[
    [
        "algoritmo",
        "criterio",
        "unidade",
        "P_escolhido",
        "custo_com_P",
        "custo_sem_caminhada",
        "ganho",
        "ganho_percentual",
        "caminhar_ajudou",
        "D_walk_m",
        "T_walk_min",
    ]
].copy()

df_ganho_ao_caminhar["custo_com_P"] = df_ganho_ao_caminhar["custo_com_P"].round(2)
df_ganho_ao_caminhar["custo_sem_caminhada"] = df_ganho_ao_caminhar["custo_sem_caminhada"].round(2)
df_ganho_ao_caminhar["ganho"] = df_ganho_ao_caminhar["ganho"].round(2)
df_ganho_ao_caminhar["ganho_percentual"] = df_ganho_ao_caminhar["ganho_percentual"].round(2)
df_ganho_ao_caminhar["D_walk_m"] = df_ganho_ao_caminhar["D_walk_m"].round(2)
df_ganho_ao_caminhar["T_walk_min"] = df_ganho_ao_caminhar["T_walk_min"].round(2)

caminho_ganhos = PASTA_SAIDA / "ganho_ao_caminhar_modelo_professor.csv"
df_ganho_ao_caminhar.to_csv(caminho_ganhos, index=False, encoding="utf-8-sig")

print(f"Tabela de ganho ao caminhar salva em: {caminho_ganhos}")
display(df_ganho_ao_caminhar)

Tabela de ganho ao caminhar salva em: saidas_modelo_professor\ganho_ao_caminhar_modelo_professor.csv


,algoritmo,criterio,unidade,P_escolhido,custo_com_P,custo_sem_caminhada,ganho,ganho_percentual,caminhar_ajudou,D_walk_m,T_walk_min
0,Dijkstra simples,menor rota em distância (do ponto P),m,P18,3679.88,3307.55,-372.33,-11.26,False,226.51,2.72
1,Dijkstra simples,rota mais rápida sem trânsito (do ponto P),s,P18,336.35,165.24,-171.11,-103.55,False,226.51,2.72
2,Dijkstra simples,rota mais rápida com trânsito sintético (do po...,s,P18,369.35,199.70,-169.65,-84.95,False,226.51,2.72
3,Dijkstra com heap,menor rota em distância (do ponto P),m,P18,3679.88,3307.55,-372.33,-11.26,False,226.51,2.72
4,Dijkstra com heap,rota mais rápida sem trânsito (do ponto P),s,P18,336.35,165.24,-171.11,-103.55,False,226.51,2.72
5,Dijkstra com heap,rota mais rápida com trânsito sintético (do po...,s,P18,369.35,199.70,-169.65,-84.95,False,226.51,2.72
6,A*,menor rota em distância (do ponto P),m,P18,3679.88,3307.55,-372.33,-11.26,False,226.51,2.72
7,A*,rota mais rápida sem trânsito (do ponto P),s,P18,336.35,165.24,-171.11,-103.55,False,226.51,2.72
8,A*,rota mais rápida com trânsito sintético (do po...,s,P18,369.35,199.70,-169.65,-84.95,False,226.51,2.72
9,Dijkstra bidirecional,menor rota em distância (do ponto P),m,P18,3679.88,3307.55,-372.33,-11.26,False,226.51,2.72


## 23. Mapa final com todos os filtros

O mapa final está organizado em dois blocos principais:

- **Casos com caminhada**: o passageiro caminha de **A** até um ponto **P**, e o carro segue de **P** até **B**;
- **Caso sem caminhada (A = P)**: o passageiro não caminha, então o ponto de embarque é o próprio **A**.

Em cada bloco existem três tipos de rota:

- rota mais rápida sem trânsito;
- rota mais rápida com trânsito sintético;
- menor rota em distância.

No cenário com trânsito, o peso `traffic_time` usa o trânsito sintético aplicado sobre trechos da rota rápida sem trânsito usada como referência. Assim, o mapa permite comparar a rota normal com a rota após penalização de congestionamento.

Esta célula não recalcula os caminhos. Ela apenas organiza, em grupos de filtros, os resultados já calculados nas células dos algoritmos. Cada camada do mapa recebe:

- nome do algoritmo;
- critério de custo;
- caminho a pé, quando houver caminhada;
- conexão curta até o ponto de embarque na malha de carro;
- rota de carro até o destino.

O objetivo é permitir comparar visualmente, no mesmo mapa, as escolhas feitas por cada algoritmo e por cada peso.

In [23]:
solucoes_mapa = {
    "Casos com caminhada": [
        ("Dijkstra simples", "rota mais rápida sem trânsito (do ponto P)", melhor_dijkstra_simples_tempo_sem, True),
        ("Dijkstra simples", "rota mais rápida com trânsito sintético (do ponto P)", melhor_dijkstra_simples_tempo_com, False),
        ("Dijkstra simples", "menor rota em distância (do ponto P)", melhor_dijkstra_simples_distancia, False),
        ("Dijkstra com heap", "rota mais rápida sem trânsito (do ponto P)", melhor_dijkstra_heap_tempo_sem, False),
        ("Dijkstra com heap", "rota mais rápida com trânsito sintético (do ponto P)", melhor_dijkstra_heap_tempo_com, False),
        ("Dijkstra com heap", "menor rota em distância (do ponto P)", melhor_dijkstra_heap_distancia, False),
        ("A*", "rota mais rápida sem trânsito (do ponto P)", melhor_astar_tempo_sem, False),
        ("A*", "rota mais rápida com trânsito sintético (do ponto P)", melhor_astar_tempo_com, False),
        ("A*", "menor rota em distância (do ponto P)", melhor_astar_distancia, False),
        ("Dijkstra bidirecional", "rota mais rápida sem trânsito (do ponto P)", melhor_bidirecional_tempo_sem, False),
        ("Dijkstra bidirecional", "rota mais rápida com trânsito sintético (do ponto P)", melhor_bidirecional_tempo_com, False),
        ("Dijkstra bidirecional", "menor rota em distância (do ponto P)", melhor_bidirecional_distancia, False),
    ],
    "Caso sem caminhada (A = P)": [
        ("Dijkstra simples", "rota mais rápida sem trânsito (do ponto A)", sem_dijkstra_simples_tempo_sem, False),
        ("Dijkstra simples", "rota mais rápida com trânsito sintético (do ponto A)", sem_dijkstra_simples_tempo_com, False),
        ("Dijkstra simples", "menor rota em distância (do ponto A)", sem_dijkstra_simples_distancia, False),
        ("Dijkstra com heap", "rota mais rápida sem trânsito (do ponto A)", sem_dijkstra_heap_tempo_sem, False),
        ("Dijkstra com heap", "rota mais rápida com trânsito sintético (do ponto A)", sem_dijkstra_heap_tempo_com, False),
        ("Dijkstra com heap", "menor rota em distância (do ponto A)", sem_dijkstra_heap_distancia, False),
        ("A*", "rota mais rápida sem trânsito (do ponto A)", sem_astar_tempo_sem, False),
        ("A*", "rota mais rápida com trânsito sintético (do ponto A)", sem_astar_tempo_com, False),
        ("A*", "menor rota em distância (do ponto A)", sem_astar_distancia, False),
        ("Dijkstra bidirecional", "rota mais rápida sem trânsito (do ponto A)", sem_bidirecional_tempo_sem, False),
        ("Dijkstra bidirecional", "rota mais rápida com trânsito sintético (do ponto A)", sem_bidirecional_tempo_com, False),
        ("Dijkstra bidirecional", "menor rota em distância (do ponto A)", sem_bidirecional_distancia, False),
    ],
}

mapa_consolidado = montar_mapa_consolidado(solucoes_mapa)

Mapa consolidado salvo em: saidas_modelo_professor\mapa_consolidado_modelo_professor.html


## 24. Validação automática das rotas

Esta célula verifica se as camadas principais fazem sentido estruturalmente.

Ela testa se:

- a rota de carro termina em **B**;
- o caso sem caminhada usa **A** como ponto de embarque;
- os casos com caminhada possuem caminho a pé de **A** até **P**;
- os custos calculados são finitos.

Se algum filtro estiver inconsistente, a célula gera erro.

Essa validação não prova que a rota é a melhor matematicamente, porque isso já é responsabilidade dos algoritmos implementados. O papel dela é detectar problemas de integração, como uma camada vazia, uma rota que não termina em **B** ou um filtro de caminhada sem caminho a pé correspondente.

In [24]:
def validar_resultado_rota(nome, criterio, resultado):
    problemas = []
    rota_p_b = resultado.get("rota_P_B")

    if not rota_p_b:
        problemas.append("rota P->B vazia")
        return problemas

    if rota_p_b[-1] != B_direcao:
        problemas.append("rota P->B não termina em B")

    if resultado.get("P") == "A":
        if rota_p_b[0] != A_direcao:
            problemas.append("caso sem caminhada não começa em A")
    else:
        no_p_caminhada = int(resultado["no_caminhada"])
        try:
            nx.shortest_path(G_caminhada, A_caminhada, no_p_caminhada, weight="length")
        except Exception as exc:
            problemas.append(f"sem caminho a pé A->P: {type(exc).__name__}")

    if resultado.get("custo") is None or not math.isfinite(float(resultado["custo"])):
        problemas.append("custo inválido")

    return problemas


linhas_validacao = []
for grupo, itens in solucoes_mapa.items():
    for algoritmo, criterio, resultado, _ in itens:
        problemas = validar_resultado_rota(algoritmo, criterio, resultado)
        linhas_validacao.append({
            "grupo": grupo,
            "algoritmo": algoritmo,
            "criterio": criterio,
            "P": resultado.get("P"),
            "status": "OK" if not problemas else "ERRO",
            "problemas": "; ".join(problemas),
        })

df_validacao_rotas = pd.DataFrame(linhas_validacao)
display(df_validacao_rotas)

if (df_validacao_rotas["status"] != "OK").any():
    raise AssertionError("Algumas camadas/filtros têm problemas de rota.")

print("VALIDACAO_FILTROS_OK")

,grupo,algoritmo,criterio,P,status,problemas
0,Casos com caminhada,Dijkstra simples,rota mais rápida sem trânsito (do ponto P),P18,OK,
1,Casos com caminhada,Dijkstra simples,rota mais rápida com trânsito sintético (do po...,P18,OK,
2,Casos com caminhada,Dijkstra simples,menor rota em distância (do ponto P),P18,OK,
3,Casos com caminhada,Dijkstra com heap,rota mais rápida sem trânsito (do ponto P),P18,OK,
4,Casos com caminhada,Dijkstra com heap,rota mais rápida com trânsito sintético (do po...,P18,OK,
5,Casos com caminhada,Dijkstra com heap,menor rota em distância (do ponto P),P18,OK,
6,Casos com caminhada,A*,rota mais rápida sem trânsito (do ponto P),P18,OK,
7,Casos com caminhada,A*,rota mais rápida com trânsito sintético (do po...,P18,OK,
8,Casos com caminhada,A*,menor rota em distância (do ponto P),P18,OK,
9,Casos com caminhada,Dijkstra bidirecional,rota mais rápida sem trânsito (do ponto P),P18,OK,


VALIDACAO_FILTROS_OK


## 25. Fechamento

Este notebook resolve o modelo pedido pelo professor sem incluir posição inicial do veículo antes do embarque.

A modelagem final é:

```text
A -> P: caminhada
P -> B: carro
```

O ponto **P** é escolhido entre candidatos alcançáveis a pé dentro de **X** e próximos da malha viária. O ícone de carro no mapa mostra exatamente onde ocorre o embarque na rede de veículos.

As tabelas e o mapa final permitem comparar algoritmos, pesos, trânsito sintético, caso sem caminhada e ganho ao caminhar.